# KPIs (Entrega 2)

Calcula os KPIs a partir da tabela gerada em `preparacao_dados.ipynb` e o resultado é gravado em `src/Staging/kpis.parquet`.

- **Entrada:** `src/Staging/base_larga.parquet`.
- **Saída:** `src/Staging/kpis.parquet` 

**Convenções:** valores em R$; os sinais da base são mantidos (saídas de caixa e passivos negativos); Ano 1 = 2027 ... Ano 12 = 2038 (encerramento da concessão). As contas que não existem no Ano 12 ficam vazias, e os KPIs que dependem delas também.

## Bloco 1 - Leitura da base e identificação

O DataFrame `kpis` começa só com as colunas de identificação: `cenario` (ex.: `Cen_00001`), `cenario_num`, `ano_num` (1 a 12) e `ano_calendario` (2027 a 2038). Cada bloco seguinte acrescenta colunas de KPI a ele.

In [1]:
from pathlib import Path

import pandas as pd

PASTA_STAGING = Path("../Staging")

base = pd.read_parquet(PASTA_STAGING / "base_larga.parquet")

kpis = base[["cenario", "cenario_num", "ano_num", "ano_calendario"]].copy()

print("Base larga:", base.shape)
print("KPIs (por enquanto só identificação):", kpis.shape)

Base larga: (14400, 73)
KPIs (por enquanto só identificação): (14400, 4)


## Bloco 2 - Receita líquida, EBITDA e margem EBITDA

- `receita_liquida` = `dre_receita` + `dre_tributos`.
- `ebitda` = linha `dre_ebitda` da base da CTI
- `margem_ebitda` = `ebitda` / `receita_liquida`. 


In [3]:
kpis["receita_liquida"] = base["dre_receita"] + base["dre_tributos"]
kpis["ebitda"] = base["dre_ebitda"]
kpis["margem_ebitda"] = kpis["ebitda"] / kpis["receita_liquida"]

colunas_bloco = ["ano_calendario", "receita_liquida", "ebitda", "margem_ebitda"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,receita_liquida,ebitda,margem_ebitda
0,2027,6.251265e+09,5.188381e+09,0.829973
1,2028,5.611118e+09,4.630952e+09,0.825317
2,2029,5.721581e+09,4.883290e+09,0.853486
3,2030,5.949170e+09,5.148066e+09,0.865342
4,2031,6.255466e+09,5.453240e+09,0.871756
5,2032,6.644719e+09,5.834764e+09,0.878105
6,2033,7.053177e+09,6.236914e+09,0.884270
7,2034,7.536001e+09,6.693782e+09,0.888241
8,2035,8.000228e+09,7.136917e+09,0.892089
9,2036,8.529061e+09,7.653142e+09,0.897302


## Bloco 3 - Lucro líquido e saldo final de caixa

- `lucro_liquido` = `dre_resultado_liquido`
- `saldo_final_caixa` = `flu_saldo_final`


In [4]:
kpis["lucro_liquido"] = base["dre_resultado_liquido"]
kpis["saldo_final_caixa"] = base["flu_saldo_final"]

colunas_bloco = ["ano_calendario", "lucro_liquido", "saldo_final_caixa"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,lucro_liquido,saldo_final_caixa
0,2027,2.443243e+09,1.205265e+09
1,2028,2.000834e+09,1.022427e+09
2,2029,2.123146e+09,1.748421e+09
3,2030,2.318127e+09,1.117390e+09
4,2031,2.642606e+09,3.073864e+08
5,2032,2.955369e+09,2.661750e+08
6,2033,3.153307e+09,4.276930e+08
7,2034,3.428037e+09,1.369964e+09
8,2035,3.734760e+09,1.923333e+09
9,2036,4.238241e+09,2.325378e+09


## Bloco 4 - EBIT, capital investido e EVA

`WACC` = 10% ao ano e `ALIQUOTA_IR` = 34% (IR/CS aplicado ao EBIT).

- `ebit` = `dre_resultado_operacional`.
- `capital_investido` = -(`bal_patrimonio_liquido` + `bal_emprestimos` + `bal_emprest`)
- `eva` = `ebit` x (1 - `ALIQUOTA_IR`) - `capital_investido` x `WACC`.

In [7]:
WACC = 0.10
ALIQUOTA_IR = 0.34

patrimonio_liquido = base["bal_patrimonio_liquido"].fillna(0)
emprestimos_cp = base["bal_emprestimos"].fillna(0)
emprestimos_lp = base["bal_emprest"].fillna(0)

kpis["ebit"] = base["dre_resultado_operacional"]
kpis["capital_investido"] = -(patrimonio_liquido + emprestimos_cp + emprestimos_lp)
kpis["eva"] = kpis["ebit"] * (1 - ALIQUOTA_IR) - kpis["capital_investido"] * WACC

colunas_bloco = ["ano_calendario", "ebit", "capital_investido", "eva"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,ebit,capital_investido,eva
0,2027,4.286529e+09,7.942794e+09,2.034829e+09
1,2028,3.549559e+09,7.439164e+09,1.598792e+09
2,2029,3.709887e+09,7.533471e+09,1.695178e+09
3,2030,3.899047e+09,6.422887e+09,1.931082e+09
4,2031,4.298228e+09,5.403774e+09,2.296453e+09
5,2032,4.657110e+09,4.740097e+09,2.599683e+09
6,2033,4.913431e+09,4.044248e+09,2.838440e+09
7,2034,5.303891e+09,3.947081e+09,3.105860e+09
8,2035,5.683662e+09,3.884704e+09,3.362747e+09
9,2036,6.319144e+09,3.955801e+09,3.775055e+09


## Bloco 5 - Fluxo de caixa direto: recebimentos, pagamentos e FCO

- `dfc_recebimentos` = `flu_receita`
- `dfc_pagamentos` = `flu_tributos` + `flu_custos` + `flu_imposto_de_renda_e_contribuicao_social`
- `dfc_fco` = `dfc_recebimentos` + `dfc_pagamentos`.

In [8]:
kpis["dfc_recebimentos"] = base["flu_receita"]
kpis["dfc_pagamentos"] = (
    base["flu_tributos"]
    + base["flu_custos"]
    + base["flu_imposto_de_renda_e_contribuicao_social"]
)
kpis["dfc_fco"] = kpis["dfc_recebimentos"] + kpis["dfc_pagamentos"]

colunas_bloco = ["ano_calendario", "dfc_recebimentos", "dfc_pagamentos", "dfc_fco"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,dfc_recebimentos,dfc_pagamentos,dfc_fco
0,2027,5.396831e+09,-2.169367e+09,3.227463e+09
1,2028,5.872798e+09,-2.624139e+09,3.248659e+09
2,2029,6.123728e+09,-2.691079e+09,3.432649e+09
3,2030,6.443489e+09,-3.186545e+09,3.256944e+09
4,2031,6.812088e+09,-3.591245e+09,3.220842e+09
5,2032,7.237617e+09,-3.315415e+09,3.922202e+09
6,2033,7.710044e+09,-3.471163e+09,4.238881e+09
7,2034,8.242633e+09,-3.730966e+09,4.511667e+09
8,2035,8.742053e+09,-4.379581e+09,4.362472e+09
9,2036,9.313009e+09,-4.920874e+09,4.392134e+09


## Bloco 6 - Fluxo de caixa direto: investimento, financiamento e caixa


- `dfc_investimentos` = `flu_investimentos` 
- `dfc_juros_liquidos` = `flu_resultado_financeiro`
- `dfc_dividendos` = `flu_distribuicao_para_acionista` 
- `dfc_financiamento` = `dfc_juros_liquidos` + `dfc_dividendos` 
- `dfc_variacao_liquida` = `dfc_fco` + `dfc_investimentos` + `dfc_financiamento`.
- `dfc_caixa_inicial` = `flu_saldo_inicial`
- `dfc_caixa_final` = `dfc_caixa_inicial` + `dfc_variacao_liquida`.



In [10]:
kpis["dfc_investimentos"] = base["flu_investimentos"]
kpis["dfc_juros_liquidos"] = base["flu_resultado_financeiro"]
kpis["dfc_dividendos"] = base["flu_distribuicao_para_acionista"]
kpis["dfc_financiamento"] = kpis["dfc_juros_liquidos"] + kpis["dfc_dividendos"]
kpis["dfc_variacao_liquida"] = (
    kpis["dfc_fco"] + kpis["dfc_investimentos"] + kpis["dfc_financiamento"]
)
kpis["dfc_caixa_inicial"] = base["flu_saldo_inicial"]
kpis["dfc_caixa_final"] = kpis["dfc_caixa_inicial"] + kpis["dfc_variacao_liquida"]

colunas_bloco = [
    "ano_calendario",
    "dfc_investimentos",
    "dfc_financiamento",
    "dfc_variacao_liquida",
    "dfc_caixa_inicial",
    "dfc_caixa_final",
]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,dfc_investimentos,dfc_financiamento,dfc_variacao_liquida,dfc_caixa_inicial,dfc_caixa_final
0,2027,-5.340012e+08,-2.262812e+09,4.306498e+08,7.746152e+08,1.205265e+09
1,2028,-3.320350e+08,-3.099461e+09,-1.828375e+08,1.205265e+09,1.022427e+09
2,2029,-1.481845e+08,-2.558471e+09,7.259936e+08,1.022427e+09,1.748421e+09
3,2030,-8.644133e+07,-3.801533e+09,-6.310307e+08,1.748421e+09,1.117390e+09
4,2031,-4.467833e+07,-3.986168e+09,-8.100039e+08,1.117390e+09,3.073864e+08
5,2032,-8.998893e+07,-3.873424e+09,-4.121136e+07,3.073864e+08,2.661750e+08
6,2033,-5.248761e+07,-4.024876e+09,1.615180e+08,2.661750e+08,4.276930e+08
7,2034,-3.433239e+07,-3.535064e+09,9.422711e+08,4.276930e+08,1.369964e+09
8,2035,-1.412965e+08,-3.667807e+09,5.533691e+08,1.369964e+09,1.923333e+09
9,2036,-5.410245e+07,-3.935987e+09,4.020450e+08,1.923333e+09,2.325378e+09


## Bloco 7 - Índices da DFC: conversão, FCL, FCO/EBITDA e FCO/Lucro líquido

- `taxa_conversao` = `dfc_recebimentos` / `dre_receita`
- `fcl` = `dfc_fco` + `dfc_investimentos`
- `fco_ebitda` = `dfc_fco` / `ebitda`
- `fco_lucro_liquido` = `dfc_fco` / `lucro_liquido`


In [11]:
kpis["taxa_conversao"] = kpis["dfc_recebimentos"] / base["dre_receita"]
kpis["fcl"] = kpis["dfc_fco"] + kpis["dfc_investimentos"]
kpis["fco_ebitda"] = kpis["dfc_fco"] / kpis["ebitda"]
kpis["fco_lucro_liquido"] = kpis["dfc_fco"] / kpis["lucro_liquido"]

colunas_bloco = ["ano_calendario", "taxa_conversao", "fcl", "fco_ebitda", "fco_lucro_liquido"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,taxa_conversao,fcl,fco_ebitda,fco_lucro_liquido
0,2027,0.802774,2.693462e+09,0.622056,1.320975
1,2028,0.960402,2.916624e+09,0.701510,1.623652
2,2029,0.979595,3.284465e+09,0.702938,1.616775
3,2030,0.990311,3.170502e+09,0.632654,1.404989
4,2031,0.995234,3.176164e+09,0.590629,1.218813
5,2032,0.995440,3.832213e+09,0.672213,1.327144
6,2033,0.998698,4.186394e+09,0.679644,1.344265
7,2034,0.999230,4.477335e+09,0.674009,1.316108
8,2035,0.998360,4.221176e+09,0.611254,1.168073
9,2036,0.997683,4.338032e+09,0.573899,1.036311


## Bloco 8 - Índices da DFC: capex, cobertura de juros, payout e cobertura de dividendos

- `capex_receita` = `dfc_investimentos`/ `dre_receita`.
- `capex_da` = `dfc_investimentos` / `dre_depreciacao_e_amortizacao`
- `cobertura_juros_caixa` = `dfc_fco` / `flu_despesas_financeiras`
- `payout_caixa` = |`dfc_dividendos` / `lucro_liquido`
- `cobertura_dividendos` = `fcl` / `dfc_dividendos`

In [13]:
investimentos_pagos = kpis["dfc_investimentos"].abs()
dividendos_pagos = kpis["dfc_dividendos"].abs()
juros_pagos = base["flu_despesas_financeiras"].abs()
depreciacao_amortizacao = base["dre_depreciacao_e_amortizacao"].abs()

kpis["capex_receita"] = investimentos_pagos / base["dre_receita"]
kpis["capex_da"] = investimentos_pagos / depreciacao_amortizacao
kpis["cobertura_juros_caixa"] = kpis["dfc_fco"] / juros_pagos
kpis["payout_caixa"] = dividendos_pagos / kpis["lucro_liquido"]
kpis["cobertura_dividendos"] = kpis["fcl"] / dividendos_pagos.replace(0, float("nan"))

colunas_bloco = [
    "ano_calendario",
    "capex_receita",
    "capex_da",
    "cobertura_juros_caixa",
    "payout_caixa",
    "cobertura_dividendos",
]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,capex_receita,capex_da,cobertura_juros_caixa,payout_caixa,cobertura_dividendos
0,2027,0.079432,0.884736,3.926204,0.627202,1.757667
1,2028,0.054299,0.455924,4.658948,1.276341,1.142096
2,2029,0.023705,0.190287,5.735611,0.974845,1.586899
3,2030,0.013285,0.106199,1.842388,0.957944,1.427745
4,2031,0.006527,0.053871,1.959357,0.931011,1.290968
5,2032,0.012377,0.106196,3.426616,0.934086,1.388197
6,2033,0.006799,0.060292,3.990293,0.948256,1.400065
7,2034,0.004162,0.038598,11.911417,0.933650,1.398911
8,2035,0.016136,0.156229,12.188586,0.924226,1.222904
9,2036,0.005796,0.058008,13.050415,0.896271,1.142005


## Bloco 9 - Capital de giro: prazos médios e ciclo financeiro

Fórmulas do material 10 do professor, com base de 360 dias (`DIAS_ANO`)

- `pmr` (prazo médio de recebimento) = `bal_contas_a_receber_clientes` / `dre_receita` x `DIAS_ANO`
- `pmp` (prazo médio de pagamento) = |`bal_fornecedores`| / |`dre_custos`| x `DIAS_ANO`
- `ciclo_financeiro` = `pmr` - `pmp`
No Ano 12 a conta de fornecedores não existe na base (encerramento da concessão), então `pmp` e `ciclo_financeiro` ficam vazios nesse ano.

In [20]:
DIAS_ANO = 360

kpis["pmr"] = base["bal_contas_a_receber_clientes"] / base["dre_receita"] * DIAS_ANO
kpis["pmp"] = base["bal_fornecedores"].abs() / base["dre_custos"].abs() * DIAS_ANO
kpis["ciclo_financeiro"] = kpis["pmr"] - kpis["pmp"]

colunas_bloco = ["ano_calendario", "pmr", "pmp", "ciclo_financeiro"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,pmr,pmp,ciclo_financeiro
0,2027,2.430464e+01,17.009011,7.295633
1,2028,2.387669e+01,17.595182,6.281508
2,2029,2.335592e+01,26.309410,-2.953492
3,2030,2.243967e+01,30.878378,-8.438704
4,2031,2.133099e+01,37.895495,-16.564506
5,2032,2.008101e+01,40.390535,-20.309526
6,2033,1.891226e+01,38.539064,-19.626804
7,2034,1.769968e+01,38.494298,-20.794617
8,2035,1.667400e+01,40.612428,-23.938429
9,2036,1.564115e+01,51.398012,-35.756860


## Bloco 10 - Catálogo de KPIs e percentis

Gera dois arquivos auxiliares em `src/Staging`, usados pelo dashboard.

**`kpi_catalogo.csv`:** uma linha por KPI, com `coluna`, `nome`, `grupo`, `formula`, `unidade` e `direcao`. O dashboard monta seletores e legendas a partir dele: um KPI novo entra com uma coluna em `kpis` e uma linha no catálogo. O notebook para com erro se algum KPI ficar fora do catálogo.

- `unidade`: `R$`, `%` (gravado como fração -> 0,83), `x` (vezes) ou `dias`.
- `direcao`: se maior ou menor é melhor. `neutro` quando depende da política ou do contexto (payout, capex, dividendos e as saídas de caixa).

**`percentis.csv`:** P5, P25, P50, P75 e P95 de cada KPI, ano a ano, entre os 1.200 cenários(`kpi`, `ano_num`, `ano_calendario`, `p5`, `p25`, `p50`, `p75`, `p95`). Mostra a faixa de resultados possíveis em cada ano. Fica vazio onde o KPI não existe (PMP e ciclo financeiro no Ano 12).

A última linha mostra os percentis do saldo final de caixa, que ilustram o risco de caixa dos anos 5 a 7.

In [21]:
principais = [
    ("receita_liquida", "Receita líquida", "KPIs principais", "dre_receita + dre_tributos", "R$", "maior é melhor"),
    ("ebitda", "EBITDA", "KPIs principais", "dre_ebitda (linha da base)", "R$", "maior é melhor"),
    ("margem_ebitda", "Margem EBITDA", "KPIs principais", "ebitda / receita_liquida", "%", "maior é melhor"),
    ("lucro_liquido", "Lucro líquido", "KPIs principais", "dre_resultado_liquido", "R$", "maior é melhor"),
    ("saldo_final_caixa", "Saldo final de caixa", "KPIs principais", "flu_saldo_final", "R$", "maior é melhor"),
    ("eva", "EVA", "KPIs principais", "ebit x (1 - ALIQUOTA_IR) - capital_investido x WACC", "R$", "maior é melhor"),
]
auxiliares = [
    ("ebit", "EBIT", "Auxiliares", "dre_resultado_operacional", "R$", "maior é melhor"),
    ("capital_investido", "Capital investido", "Auxiliares", "-(PL + empréstimos CP + empréstimos LP)", "R$", "neutro"),
]
fluxo_de_caixa = [
    ("dfc_recebimentos", "Recebimentos", "Fluxo de caixa direto", "flu_receita", "R$", "maior é melhor"),
    ("dfc_pagamentos", "Pagamentos operacionais", "Fluxo de caixa direto", "flu_tributos + flu_custos + flu_imposto_de_renda_e_contribuicao_social", "R$", "neutro"),
    ("dfc_fco", "Caixa operacional (FCO)", "Fluxo de caixa direto", "dfc_recebimentos + dfc_pagamentos", "R$", "maior é melhor"),
    ("dfc_investimentos", "Investimentos", "Fluxo de caixa direto", "flu_investimentos", "R$", "neutro"),
    ("dfc_juros_liquidos", "Juros líquidos", "Fluxo de caixa direto", "flu_resultado_financeiro", "R$", "neutro"),
    ("dfc_dividendos", "Dividendos", "Fluxo de caixa direto", "flu_distribuicao_para_acionista", "R$", "neutro"),
    ("dfc_financiamento", "Fluxo de financiamento", "Fluxo de caixa direto", "dfc_juros_liquidos + dfc_dividendos", "R$", "neutro"),
    ("dfc_variacao_liquida", "Variação líquida do caixa", "Fluxo de caixa direto", "dfc_fco + dfc_investimentos + dfc_financiamento", "R$", "maior é melhor"),
    ("dfc_caixa_inicial", "Caixa inicial", "Fluxo de caixa direto", "flu_saldo_inicial", "R$", "neutro"),
    ("dfc_caixa_final", "Caixa final", "Fluxo de caixa direto", "dfc_caixa_inicial + dfc_variacao_liquida", "R$", "maior é melhor"),
]
indices_dfc = [
    ("taxa_conversao", "Taxa de conversão", "Índices da DFC", "dfc_recebimentos / dre_receita", "%", "maior é melhor"),
    ("fcl", "Fluxo de caixa livre (FCL)", "Índices da DFC", "dfc_fco + dfc_investimentos", "R$", "maior é melhor"),
    ("fco_ebitda", "FCO / EBITDA", "Índices da DFC", "dfc_fco / ebitda", "x", "maior é melhor"),
    ("fco_lucro_liquido", "FCO / Lucro líquido", "Índices da DFC", "dfc_fco / lucro_liquido", "x", "maior é melhor"),
    ("capex_receita", "Capex / Receita", "Índices da DFC", "|dfc_investimentos| / dre_receita", "%", "neutro"),
    ("capex_da", "Capex / D&A", "Índices da DFC", "|dfc_investimentos| / |dre_depreciacao_e_amortizacao|", "x", "neutro"),
    ("cobertura_juros_caixa", "Cobertura de juros em caixa", "Índices da DFC", "dfc_fco / |flu_despesas_financeiras|", "x", "maior é melhor"),
    ("payout_caixa", "Payout em caixa", "Índices da DFC", "|dfc_dividendos| / lucro_liquido", "%", "neutro"),
    ("cobertura_dividendos", "Cobertura de dividendos", "Índices da DFC", "fcl / |dfc_dividendos|", "x", "maior é melhor"),
]
capital_de_giro = [
    ("pmr", "Prazo médio de recebimento (PMR)", "Capital de giro", "bal_contas_a_receber_clientes / dre_receita x 360", "dias", "menor é melhor"),
    ("pmp", "Prazo médio de pagamento (PMP)", "Capital de giro", "|bal_fornecedores| / |dre_custos| x 360", "dias", "neutro"),
    ("ciclo_financeiro", "Ciclo financeiro", "Capital de giro", "pmr - pmp", "dias", "menor é melhor"),
]

colunas_catalogo = ["coluna", "nome", "grupo", "formula", "unidade", "direcao"]
catalogo = pd.DataFrame(
    principais + auxiliares + fluxo_de_caixa + indices_dfc + capital_de_giro,
    columns=colunas_catalogo,
)

assert set(catalogo["coluna"]) == set(kpis.columns[4:]), "algum KPI está fora do catálogo"

catalogo.to_csv(PASTA_STAGING / "kpi_catalogo.csv", index=False)

print("Catálogo:", catalogo.shape, "-> kpi_catalogo.csv")
print(catalogo["grupo"].value_counts().to_string())

Catálogo: (30, 6) -> kpi_catalogo.csv
grupo
Fluxo de caixa direto    10
Índices da DFC            9
KPIs principais           6
Capital de giro           3
Auxiliares                2


In [22]:
QUANTIS = [0.05, 0.25, 0.50, 0.75, 0.95]
NOMES_QUANTIS = ["p5", "p25", "p50", "p75", "p95"]

tabelas = []
for coluna in catalogo["coluna"]:
    por_ano = kpis.groupby("ano_num")[coluna].quantile(QUANTIS).unstack()
    por_ano.columns = NOMES_QUANTIS
    por_ano.insert(0, "kpi", coluna)
    tabelas.append(por_ano.reset_index())

percentis = pd.concat(tabelas, ignore_index=True)

anos = kpis[["ano_num", "ano_calendario"]].drop_duplicates()
percentis = percentis.merge(anos, on="ano_num")
percentis = percentis[["kpi", "ano_num", "ano_calendario"] + NOMES_QUANTIS]

percentis.to_csv(PASTA_STAGING / "percentis.csv", index=False)

print("Percentis:", percentis.shape, "-> percentis.csv")
percentis[percentis["kpi"] == "saldo_final_caixa"].round(0)

Percentis: (360, 8) -> percentis.csv


,kpi,ano_num,ano_calendario,p5,p25,p50,p75,p95
48,saldo_final_caixa,1,2027,6.776329e+08,1.180710e+09,1.190424e+09,1.201923e+09,1.218971e+09
49,saldo_final_caixa,2,2028,1.525229e+08,9.735726e+08,9.957230e+08,1.018701e+09,1.052260e+09
50,saldo_final_caixa,3,2029,7.417047e+08,1.702772e+09,1.732541e+09,1.757542e+09,1.792975e+09
51,saldo_final_caixa,4,2030,1.173867e+08,1.068549e+09,1.098968e+09,1.125201e+09,1.162722e+09
52,saldo_final_caixa,5,2031,1.254790e+05,2.588121e+08,2.937462e+08,3.245587e+08,3.642032e+08
53,saldo_final_caixa,6,2032,1.254790e+05,2.273524e+08,2.624299e+08,2.930737e+08,3.388464e+08
54,saldo_final_caixa,7,2033,1.254790e+05,3.884550e+08,4.273086e+08,4.585643e+08,5.103256e+08
55,saldo_final_caixa,8,2034,5.990701e+08,1.303315e+09,1.352820e+09,1.395323e+09,1.462476e+09
56,saldo_final_caixa,9,2035,6.737974e+08,1.776543e+09,1.846034e+09,1.922361e+09,2.026626e+09
57,saldo_final_caixa,10,2036,7.284962e+08,2.027450e+09,2.135650e+09,2.268481e+09,2.446750e+09


## Salvando os KPIs


In [9]:
destino = PASTA_STAGING / "kpis.parquet"
kpis.to_parquet(destino, index=False)

print("Salvo em:", destino.resolve())
print("Formato:", kpis.shape)
print("Tamanho: %.2f MB" % (destino.stat().st_size / 1e6))

Salvo em: /Users/harryzhu/Projects/Projeto7/src/Staging/kpis.parquet
Formato: (14400, 15)
Tamanho: 1.45 MB
